# 🗂️ LAB D11 – Gestión de Metadatos
## Fundamentos de Gestión de Datos | TECSUP 2026-I
**Docente:** Mg. Pilar Rocío Sayán Mejía  
**Semana:** 11 | **Unidad:** III – Gestión y Calidad del Dato


## 🎯 Capacidad a Lograr
Implementar un catálogo de metadatos con estándares Dublin Core e ISO 11179, aplicando trazabilidad de datos en un sistema de base de datos SQLite.


---
## ⚙️ Configuración Inicial – Conexión a la Base de Datos


In [ ]:
# Importacion de librerias necesarias
import sqlite3
import pandas as pd
import os
import json
from datetime import datetime

print("Librerias importadas correctamente")
print(f"Pandas version: {pd.__version__}")

In [ ]:
# CONEXION A LA BASE DE DATOS DEL PROYECTO
# Opcion A: usar la base real del proyecto
# DB_PATH = "/content/drive/MyDrive/FGD/mi_proyecto.db"

# Opcion B: base de ejemplo para que el laboratorio corra sin archivos externos
DB_PATH = ":memory:"

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.executescript("""
DROP TABLE IF EXISTS ventas;
DROP TABLE IF EXISTS productos;
DROP TABLE IF EXISTS clientes;

CREATE TABLE clientes (
    id_cliente INTEGER PRIMARY KEY,
    nombre TEXT NOT NULL,
    email TEXT UNIQUE,
    telefono TEXT,
    ciudad TEXT,
    fecha_registro TEXT DEFAULT CURRENT_DATE
);

CREATE TABLE productos (
    id_producto INTEGER PRIMARY KEY,
    nombre TEXT NOT NULL,
    categoria TEXT,
    precio REAL CHECK(precio > 0),
    stock INTEGER DEFAULT 0,
    proveedor TEXT
);

CREATE TABLE ventas (
    id_venta INTEGER PRIMARY KEY,
    id_cliente INTEGER REFERENCES clientes(id_cliente),
    id_producto INTEGER REFERENCES productos(id_producto),
    cantidad INTEGER,
    fecha_venta TEXT DEFAULT CURRENT_DATE,
    monto_total REAL
);
""")

cursor.executescript("""
INSERT INTO clientes VALUES (1, 'Ana Torres', 'ana@mail.com', '999111222', 'Lima', '2025-01-10');
INSERT INTO clientes VALUES (2, 'Luis Rios', 'luis@mail.com', '988333444', 'Arequipa', '2025-02-05');
INSERT INTO clientes VALUES (3, 'Maria Paz', 'maria@mail.com', '977555666', 'Cusco', '2025-03-12');

INSERT INTO productos VALUES (1, 'Laptop', 'Tecnologia', 3200.0, 15, 'TechPeru SAC');
INSERT INTO productos VALUES (2, 'Mouse', 'Accesorios', 45.0, 100, 'PerifericosSA');
INSERT INTO productos VALUES (3, 'Teclado', 'Accesorios', 120.0, 60, 'PerifericosSA');

INSERT INTO ventas VALUES (1, 1, 1, 1, CURRENT_DATE, 3200.0);
INSERT INTO ventas VALUES (2, 2, 2, 3, CURRENT_DATE, 135.0);
INSERT INTO ventas VALUES (3, 3, 3, 2, CURRENT_DATE, 240.0);
""")
conn.commit()

print("Conexion establecida y datos cargados")

---
## 📋 Actividad 1 – Marco Conceptual de Metadatos

Completa la siguiente tabla con tus propias palabras basándote en los conceptos revisados en clase:

| N° | Concepto | Definición | Ejemplo aplicado |
|----|----------|------------|-----------------|
| 1 | Metadato | | |
| 2 | Dublin Core | | |
| 3 | ISO 11179 | | |
| 4 | Catálogo de datos | | |
| 5 | Trazabilidad | | |
| 6 | Data Lineage | | |
| 7 | Esquema de metadatos | | |
| 8 | Gobernanza de metadatos | | |

> 💡 **Tip:** Un metadato es "dato sobre el dato". Piensa cómo cada estándar organiza esa información.


---
## 🛠️ Actividad 2 – Implementación del Catálogo de Metadatos

### Paso 1: Explorar la Estructura de la Base de Datos


In [ ]:
# PASO 1: Explorar tablas y columnas disponibles
print("=" * 60)
print("EXPLORACION DE LA BASE DE DATOS")
print("=" * 60)

cursor.execute("""
    SELECT name
    FROM sqlite_schema
    WHERE type = 'table'
      AND name NOT LIKE 'sqlite_%'
    ORDER BY name
""")
tablas = [row[0] for row in cursor.fetchall()]
print(f"Tablas encontradas: {tablas}")

for tabla in tablas:
    print("-" * 50)
    print(f"Tabla: {tabla.upper()}")
    cursor.execute(f"PRAGMA table_info({tabla})")
    cols = cursor.fetchall()
    df_cols = pd.DataFrame(cols, columns=['cid', 'name', 'type', 'notnull', 'default_value', 'pk'])
    display(df_cols[['name', 'type', 'notnull', 'pk']])
    cursor.execute(f"SELECT COUNT(*) FROM {tabla}")
    n = cursor.fetchone()[0]
    print(f"Registros: {n}")

### Paso 2: Construir el Catálogo de Metadatos (ISO 11179 + Dublin Core)


In [ ]:
# PASO 2: Construir catalogo de metadatos completo
print("=" * 60)
print("CATALOGO DE METADATOS - ISO 11179 + DUBLIN CORE")
print("=" * 60)

hoy = datetime.now().strftime("%Y-%m-%d")


def inferir_dominio(nombre, tipo):
    nombre_l = nombre.lower()
    tipo_u = (tipo or "").upper()
    if "precio" in nombre_l or "monto" in nombre_l or "total" in nombre_l:
        return "Numerico positivo"
    if "fecha" in nombre_l or "date" in nombre_l:
        return "Fecha ISO 8601"
    if "email" in nombre_l:
        return "Correo electronico"
    if nombre_l.startswith("id_") or nombre_l == "id":
        return "Identificador entero"
    if "INT" in tipo_u:
        return "Entero"
    if "REAL" in tipo_u or "FLOAT" in tipo_u or "NUM" in tipo_u:
        return "Decimal"
    return "Texto"


def tipo_dublin_core(tipo):
    tipo_u = (tipo or "").upper()
    if "INT" in tipo_u:
        return "Integer"
    if "REAL" in tipo_u or "FLOAT" in tipo_u or "NUM" in tipo_u:
        return "Decimal"
    if "DATE" in tipo_u:
        return "Date"
    return "Text"


def obtener_completitud(df_tabla, columna):
    total_filas = len(df_tabla)
    if total_filas == 0 or columna not in df_tabla.columns:
        return 100.0
    nulos = df_tabla[columna].isna().sum()
    return round((1 - nulos / total_filas) * 100, 1)


catalogo_registros = []

for tabla in tablas:
    columnas = pd.read_sql_query(f"PRAGMA table_info({tabla})", conn)
    df_tabla = pd.read_sql_query(f"SELECT * FROM {tabla}", conn)
    total_filas = len(df_tabla)

    for _, col in columnas.iterrows():
        nombre_campo = col["name"]
        tipo_dato = col["type"] if col["type"] else "TEXT"
        es_pk = int(col["pk"]) == 1
        es_not_null = int(col["notnull"]) == 1

        catalogo_registros.append({
            "tabla": tabla,
            "nombre_campo": nombre_campo,
            "identificador": f"{tabla}.{nombre_campo}",
            "tipo_dato": tipo_dato,
            "es_clave": "PK" if es_pk else ("NOT NULL" if es_not_null else "Opcional"),
            "descripcion": f"Campo {nombre_campo} de la entidad {tabla}",
            "dominio": inferir_dominio(nombre_campo, tipo_dato),
            "dc_identifier": f"{tabla}.{nombre_campo}",
            "dc_type": tipo_dublin_core(tipo_dato),
            "dc_source": "Base de datos del proyecto FGD",
            "dc_date": hoy,
            "dc_rights": "Uso academico - TECSUP 2026",
            "completitud_pct": obtener_completitud(df_tabla, nombre_campo),
            "total_registros": total_filas,
            "origen": "Carga inicial del proyecto",
            "ultima_revision": hoy,
        })

df_catalogo = pd.DataFrame(catalogo_registros)

print()
print(f"Catalogo generado: {len(df_catalogo)} campos documentados")
print(f"Tablas cubiertas: {df_catalogo['tabla'].nunique()}")
print("Vista previa:")
display(df_catalogo[['tabla', 'nombre_campo', 'tipo_dato', 'es_clave', 'dominio', 'completitud_pct']])

### Paso 3: Análisis por Tipo de Dato


In [ ]:
# PASO 3: Analisis estadistico por tipo de dato
print("=" * 60)
print("ANALISIS POR TIPO DE DATO")
print("=" * 60)

resumen = df_catalogo.groupby(['tabla', 'dc_type']).agg(
    campos=('nombre_campo', 'count'),
    completitud_promedio=('completitud_pct', 'mean')
).reset_index()

display(resumen)

print("Completitud promedio global por tabla:")
comp_tabla = df_catalogo.groupby('tabla')['completitud_pct'].mean().round(1)
for tabla, pct in comp_tabla.items():
    barra = "#" * int(pct / 5) + "." * (20 - int(pct / 5))
    print(f"{tabla:15s} [{barra}] {pct}%")

### Paso 4: Exportar Catálogo a CSV y Ficha Técnica


In [ ]:
# PASO 4: Exportar catalogo de metadatos
import io

csv_buffer = io.StringIO()
df_catalogo.to_csv(csv_buffer, index=False, encoding='utf-8')
csv_content = csv_buffer.getvalue()

print("Vista previa del CSV exportado:")
for i, linea in enumerate(csv_content.splitlines()[:5]):
    print(f"  {linea}")
print(f"  ... ({len(df_catalogo)} filas en total)")

try:
    with open('/content/catalogo_metadatos.csv', 'w', encoding='utf-8') as f:
        f.write(csv_content)
    print("Archivo guardado: /content/catalogo_metadatos.csv")
except OSError:
    df_catalogo.to_csv('catalogo_metadatos.csv', index=False, encoding='utf-8')
    print("Archivo guardado: catalogo_metadatos.csv")

ficha_tecnica = {
    "proyecto": "Fundamentos de Gestion de Datos - FGD",
    "version_catalogo": "1.0",
    "fecha_creacion": hoy,
    "autor": "Pilar Rocio Sayan Mejia",
    "estandares_aplicados": ["ISO 11179", "Dublin Core"],
    "tablas_documentadas": tablas,
    "total_campos": len(df_catalogo),
    "completitud_global": round(df_catalogo['completitud_pct'].mean(), 1)
}

print("Ficha tecnica del catalogo:")
print(json.dumps(ficha_tecnica, indent=2, ensure_ascii=False))

### Paso 5: Trazabilidad y Linaje de Datos


In [ ]:
# PASO 5: Registro de linaje de datos
print("=" * 60)
print("TRAZABILIDAD Y LINAJE DE DATOS")
print("=" * 60)

cursor.execute("""
CREATE TABLE IF NOT EXISTS lineaje_datos (
    id_evento INTEGER PRIMARY KEY AUTOINCREMENT,
    tabla_origen TEXT,
    campo TEXT,
    operacion TEXT,
    usuario TEXT,
    fecha_hora TEXT,
    descripcion TEXT
)
""")

eventos_linaje = [
    ("clientes", "todos", "INSERT", "sistema", "Carga inicial de clientes desde CSV de ventas"),
    ("productos", "todos", "INSERT", "sistema", "Carga inicial de catalogo de productos"),
    ("ventas", "todos", "INSERT", "sistema", "Registro de transacciones del periodo"),
    ("clientes", "email", "UPDATE", "admin", "Normalizacion de correos electronicos"),
    ("productos", "precio", "UPDATE", "admin", "Actualizacion trimestral de precios"),
    ("ventas", "monto_total", "SELECT", "analista", "Generacion de reporte mensual de ventas"),
]

cursor.executemany("""
    INSERT INTO lineaje_datos (tabla_origen, campo, operacion, usuario, fecha_hora, descripcion)
    VALUES (?, ?, ?, ?, datetime('now'), ?)
""", eventos_linaje)
conn.commit()

df_linaje = pd.read_sql_query(
    "SELECT tabla_origen, campo, operacion, usuario, fecha_hora, descripcion FROM lineaje_datos",
    conn
)

display(df_linaje)
print("Operaciones por tabla:")
display(df_linaje.groupby(['tabla_origen', 'operacion']).size().reset_index(name='conteo'))

### Paso 6: Guardar Catálogo en SQLite


In [ ]:
# PASO 6: Persistir catalogo en la base de datos
print("=" * 60)
print("PERSISTENCIA DEL CATALOGO EN SQLITE")
print("=" * 60)

df_catalogo.to_sql('catalogo_metadatos', conn, if_exists='replace', index=False)
conn.commit()

cursor.execute("SELECT COUNT(*) FROM catalogo_metadatos")
n = cursor.fetchone()[0]
print(f"Tabla catalogo_metadatos creada con {n} registros")

print("Campos clave PK en la base de datos:")
df_pks = pd.read_sql_query(
    "SELECT tabla, nombre_campo, tipo_dato, dominio FROM catalogo_metadatos WHERE es_clave='PK'",
    conn
)
display(df_pks)

print("Campos con completitud menor a 100%:")
df_inc = pd.read_sql_query(
    "SELECT tabla, nombre_campo, completitud_pct FROM catalogo_metadatos WHERE completitud_pct < 100 ORDER BY completitud_pct",
    conn
)
if len(df_inc) > 0:
    display(df_inc)
else:
    print("Todos los campos tienen completitud del 100%")

---
## 💬 Actividad 3 – Análisis Reflexivo

Responde las siguientes preguntas con base en lo desarrollado en esta sesión:

**A) ¿Cuál es la diferencia entre un metadato estructural y un metadato descriptivoDa un ejemplo de cada uno con tus tablas.**

> _Tu respuesta aquí_

**B) ¿Por qué es importante el linaje de datos en un sistema empresarial¿Qué problemas puede prevenir.**

> _Tu respuesta aquí_

**C) Compara el estándar Dublin Core con ISO 11179. ¿En qué contextos aplicarías cada uno.**

> _Tu respuesta aquí_

**D) Si tu base de datos tuviera 50 tablas con 20 columnas cada una, ¿cómo automatizarías la generación del catálogo¿Qué información adicional agregarías.**

> _Tu respuesta aquí_


---
## Control de calidad del laboratorio

Este laboratorio no entrena un modelo predictivo. Por eso no existen variables `X`, `y`, `target`, `train`, `test`, metricas de accuracy ni validacion de modelo. En consecuencia, el riesgo de `data leakage` de machine learning no aplica aqui.

El control correcto para Semana 11 es otro: verificar trazabilidad, catalogo de metadatos, completitud de campos, reglas de negocio, linaje del dato y persistencia del catalogo en SQLite. Si en una semana posterior se entrena un modelo, ahi si se debe separar train/test antes de imputar, escalar o seleccionar variables.



In [ ]:
# Auditoria simple: confirmar que este notebook no entrena modelos ni usa train/test
objetos_ml = [nombre for nombre in globals() if nombre.lower() in ['x', 'y', 'target', 'x_train', 'x_test', 'y_train', 'y_test', 'modelo', 'model']]

print('Objetos tipicos de modelado encontrados:', objetos_ml)
print('Conclusion: el laboratorio trabaja metadatos y linaje; no entrena modelos predictivos.')
print('Control anti data leakage: no aplica entrenamiento, pero se deja documentado para semanas de ML.')

conn.close()
print('Conexion cerrada correctamente')

---
## ✅ Conclusiones

1. _Escribe aquí tu primera conclusión sobre la gestión de metadatos_
2. _Escribe aquí tu segunda conclusión sobre los estándares Dublin Core / ISO 11179_
3. _Escribe aquí tu tercera conclusión sobre la trazabilidad y el linaje de datos_


---
## 📚 Material Complementario

| Recurso | Enlace |
|---------|--------|
| Dublin Core Metadata Initiative | https://dublincore.org |
| ISO 11179 – Metadata Registry | https://www.iso.org/standard/61451.html |
| Data Catalog con Python | https://docs.python.org/3/library/sqlite3.html |
| DAMA-DMBOK – Gestión de Metadatos | https://www.dama.org/cpages/body-of-knowledge |
| SQLite PRAGMA table_info | https://www.sqlite.org/pragma.html#pragma_table_info |
